
# Predicción Semanal de Demanda — EDA y Entendimiento del Negocio

### Ficha del Proyecto e Integrantes

| # | Nombre del Integrante | Rol / Responsabilidad |
| :-: | :--- | :--- |
| 1 | Adrian Edwin Aguirre Rojas | Análisis EDA y Modelado |
| 2 | Yovani Andia Quispe | Coordinación general del equipo |
| 3 | Takeshi Israel Carvajal Lima | Diseño del pipeline de ingesta y limpieza de datos |
| 4 | David Lamas Crispín | Story teller encargado de: Selección, configuración y entrenamiento de los modelos de regresión (Regresión Lineal, Random Forest, XGBoost). |
| 5 | Jesus Gabriel Cruz Lavadenz | Programador: encargado del desarrollo e implementación del código del proyecto |
| 6 | Diego Rodirgo Marca Cuevas | Programador: encargado del desarrollo e implementación del código del proyecto |

## 1. Objetivo de negocio

El objetivo de negocio es **reducir los quiebres de stock causados por subestimar la demanda**, utilizando una predicción semanal para apoyar las decisiones de abastecimiento.

La idea es estimar cuántas unidades serán necesarias en una semana futura y utilizar esa predicción para planificar el inventario. Como política de negocio, se puede agregar un margen de seguridad para disminuir el riesgo de que la demanda real supere la cantidad prevista.

> **Nota:** El análisis principal se basa en el archivo `demanda_predictiva_takeshi.csv` y en los resultados documentados del proyecto. El resultado final más robusto corresponde al backtesting temporal.



## 2. Importación de librerías

Se utilizan `pandas` y `numpy` para la manipulación y cálculo de datos, `matplotlib` y `seaborn` para las visualizaciones, y herramientas de `scikit-learn` para revisar valores atípicos mediante un método estadístico sencillo.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")



## 3. Carga y descripción de los datos

La unidad de análisis es una **semana de demanda**. Las variables disponibles incluyen la fecha de la semana, la demanda real, retardos (`lag_1` a `lag_4`), variables temporales y promedios móviles de demanda.

El archivo utilizado contiene la demanda semanal ya preparada para el análisis predictivo.


In [ ]:

# Buscar el archivo en la carpeta actual y en /mnt/data
candidatos = [
    Path("demanda_predictiva_takeshi.csv"),
    Path("/mnt/data/demanda_predictiva_takeshi.csv")
]

archivo = next((p for p in candidatos if p.exists()), None)

if archivo is None:
    print("No se encontró el CSV en las rutas esperadas.")
    print("Coloca 'demanda_predictiva_takeshi.csv' junto al notebook para ejecutar todas las celdas.")
else:
    df = pd.read_csv(archivo)
    df["semana"] = pd.to_datetime(df["semana"], errors="coerce")
    df = df.sort_values("semana").reset_index(drop=True)
    print(f"Archivo cargado: {archivo}")
    print(f"Registros: {len(df)}")
    print(f"Variables: {df.shape[1]}")
    display(df.head())


In [ ]:

if 'df' in globals():
    print("Dimensiones:", df.shape)
    print("\nTipos de datos:")
    display(df.dtypes.to_frame("tipo"))

    print("\nResumen estadístico:")
    display(df.describe(include="all").T)



### Variables principales

- `semana`: fecha de la semana evaluada.
- `demanda_real`: demanda observada y variable objetivo.
- `lag_1` a `lag_4`: demanda de semanas anteriores.
- `mes`, `semana_del_anio` y `trimestre`: variables temporales.
- `promedio_movil_4_correcto` y `promedio_movil_8_correcto`: promedios de demanda histórica.

Los `lag` representan información pasada. Por ejemplo, `lag_1` corresponde a la demanda de la semana inmediatamente anterior.



## 4. Limpieza y preparación

La preparación debe conservar el orden temporal. Esto es importante porque la predicción de demanda utiliza información del pasado para predecir una semana futura.

También se revisan duplicados, valores faltantes e inconsistencias básicas antes de realizar el EDA.


In [ ]:

if 'df' in globals():
    print("Duplicados completos:", df.duplicated().sum())
    print("\nValores faltantes por variable:")
    faltantes = df.isna().sum().sort_values(ascending=False)
    display(faltantes.to_frame("faltantes"))

    print("\nFechas no válidas:", df["semana"].isna().sum())


In [ ]:

if 'df' in globals():
    # Comprobación de consistencia temporal
    df["delta_dias"] = df["semana"].diff().dt.days
    print("Intervalos entre semanas (días):")
    display(df["delta_dias"].value_counts(dropna=True).sort_index())

    # Eliminamos solo la columna auxiliar; no modificamos las variables originales
    df.drop(columns=["delta_dias"], inplace=True)



Si los datos faltantes, duplicados o inconsistencias aparecen en cero, no es necesario aplicar una transformación destructiva. Se conserva el conjunto de datos porque eliminar observaciones válidas podría afectar el análisis temporal.

Además, los promedios móviles utilizados en el proyecto deben construirse con información anterior mediante `shift(1)` para evitar fuga de información.


In [ ]:

if 'df' in globals() and "demanda_real" in df.columns:
    df["promedio_movil_4_verificado"] = (
        df["demanda_real"].shift(1).rolling(window=4).mean()
    )
    df["promedio_movil_8_verificado"] = (
        df["demanda_real"].shift(1).rolling(window=8).mean()
    )

    display(df[[
        "semana", "demanda_real",
        "promedio_movil_4_verificado",
        "promedio_movil_8_verificado"
    ]].head(10))



## 5. Definición y cálculo de la métrica principal

La métrica principal de negocio es **TQS (Tasa de Quiebre de Stock)**.

### Fórmula

**TQS = (número de semanas donde demanda_real > predicción / número total de semanas) × 100**

Esta métrica mide el porcentaje de semanas en las que se subestimó la demanda. Es importante porque el objetivo del proyecto es disminuir los quiebres de stock.

Para el análisis reproducible se calculará el TQS de la línea base usando `lag_1` como predicción. La línea base permite comparar posteriormente si un modelo más complejo realmente aporta valor.


In [ ]:

if 'df' in globals():
    df["prediccion_baseline"] = df["lag_1"]
    df["quiebre_baseline"] = (
        df["demanda_real"] > df["prediccion_baseline"]
    ).astype(int)

    tqs_baseline = df["quiebre_baseline"].mean() * 100
    mae_baseline = (
        df["demanda_real"] - df["prediccion_baseline"]
    ).abs().mean()
    wape_baseline = (
        (df["demanda_real"] - df["prediccion_baseline"]).abs().sum()
        / df["demanda_real"].sum()
    ) * 100

    print(f"TQS baseline: {tqs_baseline:.2f}%")
    print(f"MAE baseline: {mae_baseline:.2f}")
    print(f"WAPE baseline: {wape_baseline:.2f}%")



### Resultado documentado del proyecto

En la evaluación general del baseline sobre las **88 semanas** disponibles, el proyecto documenta:

- **WAPE:** 16.40%
- **TQS:** 56.82%

Estas cifras corresponden a la evaluación general del baseline y no deben confundirse con el test temporal de 16 semanas ni con el backtesting final de 40 semanas.



## 6. Respuestas a las cinco preguntas del EDA

### Pregunta 1 — ¿Qué datos tengo?

Se dispone de un conjunto de demanda semanal con **88 semanas** en la evaluación general documentada. La variable objetivo es `demanda_real`, acompañada de variables históricas (`lag_1` a `lag_4`), variables temporales y promedios móviles.

El conjunto está orientado a predecir demanda futura y, posteriormente, apoyar decisiones de abastecimiento.



### Pregunta 2 — ¿Hay datos faltantes?

Esta pregunta se responde mediante el conteo de valores `NaN` por columna mostrado anteriormente.

En el flujo del proyecto, los promedios móviles correctos generan valores faltantes iniciales de forma natural porque no existen suficientes semanas anteriores para calcular las ventanas de 4 y 8 semanas. Estos valores iniciales deben tratarse de forma coherente con el modelo temporal, evitando inventar información.


In [ ]:

if 'df' in globals():
    faltantes = df.isna().sum()
    print("Total de valores faltantes:", faltantes.sum())
    display(faltantes[faltantes > 0].to_frame("cantidad"))



### Pregunta 3 — ¿Existen valores atípicos?

Para la demanda real se puede utilizar el criterio del rango intercuartílico (IQR). Este método identifica observaciones por debajo de Q1 − 1.5×IQR o por encima de Q3 + 1.5×IQR.

Un valor atípico no debe eliminarse automáticamente: en demanda puede representar una semana real de alta venta. Por ello, primero se identifica y luego se interpreta con el contexto del negocio.


In [ ]:

if 'df' in globals():
    q1 = df["demanda_real"].quantile(0.25)
    q3 = df["demanda_real"].quantile(0.75)
    iqr = q3 - q1
    limite_inf = q1 - 1.5 * iqr
    limite_sup = q3 + 1.5 * iqr

    atipicos = df[
        (df["demanda_real"] < limite_inf) |
        (df["demanda_real"] > limite_sup)
    ]

    print(f"Q1: {q1:.2f}")
    print(f"Q3: {q3:.2f}")
    print(f"IQR: {iqr:.2f}")
    print(f"Límite inferior: {limite_inf:.2f}")
    print(f"Límite superior: {limite_sup:.2f}")
    print(f"Valores atípicos detectados: {len(atipicos)}")

    display(atipicos[["semana", "demanda_real"]].head(20))



### Pregunta 4 — ¿Cómo se distribuyen los datos?

La distribución de `demanda_real` permite observar la concentración de la demanda, su dispersión y posibles valores extremos. Esta visualización es relevante porque una demanda muy variable puede dificultar la planificación del inventario.


In [ ]:

if 'df' in globals():
    plt.figure(figsize=(10, 5))
    plt.hist(df["demanda_real"].dropna(), bins=15)
    plt.title("Distribución de la demanda real")
    plt.xlabel("Demanda real")
    plt.ylabel("Frecuencia")
    plt.tight_layout()
    plt.show()


In [ ]:

if 'df' in globals():
    plt.figure(figsize=(12, 5))
    plt.plot(df["semana"], df["demanda_real"], marker="o", linewidth=1)
    plt.title("Evolución semanal de la demanda real")
    plt.xlabel("Semana")
    plt.ylabel("Demanda real")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()



La lectura del gráfico debe centrarse en los cambios de nivel y la variabilidad semanal. Para el objetivo de negocio, los periodos de mayor demanda son especialmente importantes porque una subestimación en esas semanas puede producir quiebres de stock.



### Pregunta 5 — ¿Qué relaciones existen entre variables?

Una relación especialmente relevante es la existente entre `demanda_real` y `lag_1`, porque `lag_1` representa la demanda de la semana anterior y es precisamente la variable utilizada por la línea base.

La correlación permite medir asociación lineal, pero no demuestra causalidad.


In [ ]:

if 'df' in globals():
    variables_relacion = [
        "demanda_real", "lag_1", "lag_2", "lag_3", "lag_4",
        "promedio_movil_4_correcto", "promedio_movil_8_correcto"
    ]
    disponibles = [c for c in variables_relacion if c in df.columns]

    corr = df[disponibles].corr(numeric_only=True)

    plt.figure(figsize=(10, 7))
    sns.heatmap(corr, annot=True, fmt=".2f", center=0)
    plt.title("Correlación entre demanda y variables históricas")
    plt.tight_layout()
    plt.show()


In [ ]:

if 'df' in globals():
    plt.figure(figsize=(8, 5))
    plt.scatter(df["lag_1"], df["demanda_real"], alpha=0.7)
    plt.title("Relación entre demanda anterior y demanda real")
    plt.xlabel("Demanda de la semana anterior (lag_1)")
    plt.ylabel("Demanda real")
    plt.tight_layout()
    plt.show()



La relación entre variables históricas es útil para la predicción, pero debe interpretarse con prudencia. Una correlación alta significa asociación estadística, no que una variable cause directamente la otra.



## 7. Hallazgo interesante

El hallazgo principal del proyecto es que **el modelo predictivo por sí solo no fue suficiente para reducir los quiebres**, mientras que la combinación del Random Forest con un **factor de seguridad de 1.08** sí mostró una reducción importante en el backtesting temporal.

En las 40 semanas evaluadas mediante backtesting:

| Modelo | MAE | WAPE | TQS | Quiebres |
|---|---:|---:|---:|---:|
| Línea base | 265.62 | 15.88% | 45.00% | 18 |
| Random Forest | 334.57 | 20.00% | 45.00% | 18 |
| Random Forest + factor 1.08 | 382.66 | 22.87% | 25.00% | 10 |

Esto significa que el modelo ajustado **evitó 8 quiebres** frente a la línea base y redujo el TQS de **45% a 25%**, aunque el WAPE aumentó de **15.88% a 22.87%**.

El hallazgo muestra un **trade-off entre precisión predictiva y disponibilidad de inventario**: para el negocio puede ser razonable aceptar algo más de error si el costo de quedarse sin inventario es mayor que el costo de mantener un margen adicional.



## 8. Conclusiones

1. El problema de negocio se centra en reducir los quiebres de stock ocasionados por subestimar la demanda semanal.
2. La línea base con `lag_1` permite establecer un punto de comparación reproducible.
3. El análisis EDA permite revisar la estructura de los datos, valores faltantes, duplicados, posibles valores atípicos, distribuciones y relaciones entre variables.
4. El Random Forest inicial no mejoró la precisión respecto a la línea base en el test documentado, aunque sí redujo el TQS en ese subconjunto.
5. El resultado más importante corresponde al backtesting temporal: el modelo ajustado con un factor de seguridad de **1.08** redujo los quiebres de **18 a 10** en 40 semanas.
6. Esta mejora tuvo un costo: el WAPE aumentó de **15.88% a 22.87%**.
7. Por tanto, la solución debe entenderse como una combinación de **predicción + política de seguridad**, no como un efecto del Random Forest por sí solo.
8. No se afirma que se haya alcanzado una meta exacta de TQS de 5%, ya que los resultados documentados no llegan a ese nivel y el tamaño de las muestras no permite representar exactamente un 5% en el test de 16 semanas.



## Referencia de reproducibilidad

El notebook está organizado según la estructura solicitada: título y objetivo, librerías, carga y descripción de datos, preparación, métrica principal, cinco preguntas del EDA, hallazgo y conclusiones.

Para ejecutarlo completamente, coloca `demanda_predictiva_takeshi.csv` en la misma carpeta del notebook. El código mantiene el orden temporal y muestra las salidas de los análisis mediante tablas y gráficos.
